# 🧮 AI4Trade — Segment-Level Weighting & Ensemble Builder (Final Run)

**Objective:**  
Compute data-driven blend weights for each segment (CHN Exports, CHN Imports, USA Exports, USA Imports) using OOF sMAPE,  
then apply those weights to blend model forecasts for October 2025 at the HS6 level → aggregate to HS4.

---

### 🔁 Workflow Overview
1. **Input OOF files**  
   `predictions/oof/{model}_{segment}_h{2,3}_final.parquet`  
   Contain: `origin, destination, hs6, trade_flow, month, y_true, y_pred`

2. **Compute segment weights**
   - Aggregates OOFs from HS6 → HS4  
   - Computes **sMAPE** per validation fold  
   - Applies **fold weights** (late folds C3–C6 / U2–U5 emphasized)  
   - Derives **inverse-sMAPE weights**, clamps to [0.15 – 0.65]  
   - Blends with prior (70% data + 30% [0.45 / 0.25 / 0.30])  
   - Adds +10% recency tilt for models that win on nearest folds  
   - Saves to `/predictions/final/blend_weights_final.csv`

3. **Blend forecast files**
   - Loads `predictions/forecast/{model}_{segment}_h{2,3}_final.parquet`
   - Merges and computes weighted sum across models  
   - Saves:
     - HS6 ensemble → `ensemble_{segment}_h{2,3}_hs6_final.parquet`
     - HS4 aggregate → `ensemble_{segment}_h{2,3}_hs4_final.parquet`

4. **Merge all HS4 files**  
   Concatenate four segment outputs →  
   `/predictions/final/final_forecast_hs4_final.parquet`  
   (used by `50_make_submission.ipynb` to build the final CSV)

---

### ⚙️ Key Settings

| Parameter | Purpose | Default |
|------------|----------|----------|
| `MIX_ALPHA` | Weighting: 0.70 data-driven + 0.30 prior | 0.70 |
| `RECENCY_TILT` | +10% boost for nearest-fold winners | 1.10 |
| `W_FLOOR`, `W_CAP` | Clamp weights per model | 0.15 – 0.65 |
| `EPS` | sMAPE stabilizer (denominator) | 1e-6 |

---

### 📊 Validation References
- China (h = 2): folds C1–C6, weighted (1.0, 1.0, 1.25, 1.0, 1.5, 1.5)  
- USA (h = 3): folds U1–U5, weighted (1.0, 1.25, 1.0, 1.5, 1.5)  
- Recency folds:  
  - China → C3 (2024-10) and C6 (2025-07)  
  - USA → U2 (2024-10) and U5 (2025-07)

---

### 🚀 Run Order
`CHN-Export → CHN-Import → USA-Export → USA-Import`  
Each segment prints weighted sMAPE, recency winners, and final weights.

---

### ✅ Outputs (all suffixed `_final`)
| Category | Example Path |
|-----------|---------------|
| Weights CSV | `/predictions/final/blend_weights_final.csv` |
| HS6 Ensembles | `/predictions/final/ensemble_chn_export_h2_hs6_final.parquet` |
| HS4 Ensembles | `/predictions/final/ensemble_chn_export_h2_hs4_final.parquet` |
| Combined HS4 | `/predictions/final/final_forecast_hs4_final.parquet` |

---

**Note:**  
This notebook only blends model predictions — no re-training.  
All resulting weights and forecasts must **beat the naïve baseline** on sMAPE to qualify for submission.

In [1]:
# Colab mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# === cell: config & imports ===
import os
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/ai4trade")  # adjust if needed
OOF_DIR = BASE_DIR / "predictions" / "oof"
FORECAST_DIR = BASE_DIR / "predictions" / "forecast"
FINAL_DIR = BASE_DIR / "predictions" / "final"

FINAL_DIR.mkdir(parents=True, exist_ok=True)

# models participating in the 3-model ensemble
MODELS = [
    ("xgb_tweedie",  "xgb_tweedie"),
    ("xgb_log1p",    "xgb_log1p"),
    ("lgbm_rmse",    "lgbm_rmse"),
]

# Prior blend (used as a soft prior, not hard weights)
PRIOR_WEIGHTS = {
    "xgb_tweedie": 0.45,
    "xgb_log1p":   0.25,
    "lgbm_rmse":   0.30,
}

# Segment → horizon + fold target months + weights (from Horizon Policy)
SEGMENTS = {
    # segment_name: (horizon, {target_month: fold_weight}, recency_targets)
    "CHN_export": (2,
        {
            "2024-08-01": 1.00,  # C1
            "2024-09-01": 1.00,  # C2
            "2024-10-01": 1.25,  # C3 (mirror final)
            "2024-12-01": 1.00,  # C4
            "2025-06-01": 1.50,  # C5
            "2025-07-01": 1.50,  # C6
        },
        # recency winners to check (closest-to-target folds)
        ["2024-10-01", "2025-07-01"]
    ),
    "CHN_import": (2,
        {
            "2024-08-01": 1.00,
            "2024-09-01": 1.00,
            "2024-10-01": 1.25,
            "2024-12-01": 1.00,
            "2025-06-01": 1.50,
            "2025-07-01": 1.50,
        },
        ["2024-10-01", "2025-07-01"]
    ),
    "USA_export": (3,
        {
            "2024-08-01": 1.00,  # U1
            "2024-10-01": 1.25,  # U2 (mirror final)
            "2024-11-01": 1.00,  # U3
            "2025-06-01": 1.50,  # U4
            "2025-07-01": 1.50,  # U5
        },
        ["2024-10-01", "2025-07-01"]
    ),
    "USA_import": (3,
        {
            "2024-08-01": 1.00,
            "2024-10-01": 1.25,
            "2024-11-01": 1.00,
            "2025-06-01": 1.50,
            "2025-07-01": 1.50,
        },
        ["2024-10-01", "2025-07-01"]
    ),
}

# Floors/caps and blending knobs
W_FLOOR, W_CAP = 0.15, 0.65
MIX_ALPHA = 0.70     # data-driven weight share; 0.30 goes to prior
RECENCY_TILT = 1.10  # +10% to winners on closest folds
EPS = 1e-6

In [3]:
# === cell: utilities ===
def parse_month(x):
    # robust month parsing to YYYY-MM-DD
    return pd.to_datetime(x).strftime("%Y-%m-%d")

def smape(y_true, y_pred, epsilon=1.0):
    # sMAPE with epsilon stabilization as per our docs
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    num = np.abs(y_pred - y_true)
    den = (np.abs(y_true) + np.abs(y_pred) + epsilon) / 2.0
    return np.mean(num / den)

def ensure_columns(df, required):
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

def add_hs4(df):
    # hs6 must be string; derive hs4
    if "hs4" not in df.columns:
        df = df.copy()
        df["hs6"] = df["hs6"].astype(str).str.zfill(6)
        df["hs4"] = df["hs6"].str[:4]
    return df

def aggregate_hs6_to_hs4(df, pred_col, group_cols=("origin","destination","trade_flow","month","hs4")):
    df = add_hs4(df)
    gc = list(group_cols)
    out = (df
           .groupby(gc, as_index=False)[pred_col]
           .sum())
    return out

def load_oof_file(model, segment, horizon):
    # expected: predictions/oof/{model}_{segment}_h{2,3}_final.parquet
    path = OOF_DIR / f"{model}_{segment}_h{horizon}_final.parquet"
    if not path.exists():
        raise FileNotFoundError(f"OOF not found: {path}")
    df = pd.read_parquet(path)
    # normalize column names we rely on
    # Must have: origin, destination, hs6, trade_flow, month, y_true, y_pred (or model-specific pred col)
    if "y_pred" not in df.columns:
        # try to find a single prediction column
        pred_cols = [c for c in df.columns if c.startswith("y_pred")]
        if len(pred_cols) == 1:
            df = df.rename(columns={pred_cols[0]: "y_pred"})
        elif len(pred_cols) == 0:
            # sometimes model files store prediction as 'value' or 'pred'
            fallback = [c for c in df.columns if c in ("pred","prediction","value")]
            if len(fallback) == 1:
                df = df.rename(columns={fallback[0]: "y_pred"})
            else:
                raise ValueError(f"No prediction column found in {path}")
        else:
            raise ValueError(f"Multiple y_pred* columns in {path}: {pred_cols}")
    ensure_columns(df, ["origin","destination","hs6","trade_flow","month","y_true","y_pred"])
    df["model"] = model
    df["month"] = pd.to_datetime(df["month"]).dt.strftime("%Y-%m-%d")
    return df

def load_forecast_file(model, segment, horizon):
    # expected: predictions/forecast/{model}_{segment}_h{2,3}_final.parquet
    path = FORECAST_DIR / f"{model}_{segment}_h{horizon}_final.parquet"
    if not path.exists():
        raise FileNotFoundError(f"Forecast not found: {path}")
    df = pd.read_parquet(path)
    # normalize prediction column
    if "y_pred" not in df.columns:
        pred_cols = [c for c in df.columns if c.startswith("y_pred")]
        if len(pred_cols) == 1:
            df = df.rename(columns={pred_cols[0]: "y_pred"})
        elif len(pred_cols) == 0:
            fallback = [c for c in df.columns if c in ("pred","prediction","value")]
            if len(fallback) == 1:
                df = df.rename(columns={fallback[0]: "y_pred"})
            else:
                raise ValueError(f"No prediction column found in {path}")
        else:
            raise ValueError(f"Multiple y_pred* columns in {path}: {pred_cols}")
    ensure_columns(df, ["origin","destination","hs6","trade_flow","month","y_pred"])
    df["model"] = model
    df["month"] = pd.to_datetime(df["month"]).dt.strftime("%Y-%m-%d")
    return df

def weighted_smape_by_month(df_hs4, month_weights, model_name):
    """
    df_hs4: columns [origin,destination,trade_flow,month,hs4,y_true,y_pred,model]
    month_weights: dict YYYY-MM-DD -> weight
    """
    scores = []
    for m, w in month_weights.items():
        sub = df_hs4[df_hs4["month"] == m]
        if sub.empty:
            continue
        s = smape(sub["y_true"].values, sub["y_pred"].values, epsilon=1.0)
        scores.append((m, s, w))
    if not scores:
        return np.inf
    # weighted average
    num = sum(s * w for _, s, w in scores)
    den = sum(w for _, s, w in scores)
    return num / den

def month_winners(df_hs4, months):
    """
    Return set of model names that win (lowest sMAPE) in each given month.
    df_hs4 must contain rows for those months.
    """
    winners = set()
    for m in months:
        sub = df_hs4[df_hs4["month"] == m]
        if sub.empty:
            continue
        # compute sMAPE per model at HS4 for that month
        group = []
        for mdl, g in sub.groupby("model"):
            s = smape(g["y_true"].values, g["y_pred"].values, epsilon=1.0)
            group.append((mdl, s))
        if group:
            group.sort(key=lambda x: x[1])
            winners.add(group[0][0])
    return winners

In [4]:
# === cell: compute weights for a segment ===
def compute_segment_weights(segment):
    """
    Returns dict: {model -> final_weight} for a given segment name in SEGMENTS.
    Also returns an auxiliary dict with diagnostics (per-model weighted sMAPE, winners, etc.).
    """
    if segment not in SEGMENTS:
        raise ValueError(f"Unknown segment: {segment}")
    horizon, month_weights, recency_months = SEGMENTS[segment]
    # 1) Load and stack OOFs
    oofs = []
    for _, model in MODELS:
        df = load_oof_file(model, segment, horizon)
        oofs.append(df[["origin","destination","hs6","trade_flow","month","y_true","y_pred","model"]])
    oof_all = pd.concat(oofs, ignore_index=True)
    # 2) Aggregate to HS4 per model
    agg_list = []
    for mdl, g in oof_all.groupby("model"):
        g_hs4_true = aggregate_hs6_to_hs4(g, pred_col="y_true")
        g_hs4_pred = aggregate_hs6_to_hs4(g, pred_col="y_pred")
        # join on keys to align y_true and y_pred at HS4
        keys = ["origin","destination","trade_flow","month","hs4"]
        merged = g_hs4_true.merge(g_hs4_pred, on=keys, suffixes=("_true","_pred"))
        merged = merged.rename(columns={"y_true_true":"y_true","y_pred_pred":"y_pred"})
        merged["model"] = mdl
        agg_list.append(merged)
    hs4_all = pd.concat(agg_list, ignore_index=True)

    # 3) Compute weighted sMAPE per model across specified months
    model_scores = {}
    for mdl, g in hs4_all.groupby("model"):
        score = weighted_smape_by_month(g, month_weights, mdl)
        model_scores[mdl] = float(score)

    # 4) Convert to inverse-sMAPE data-driven weights
    inv = {m: 1.0 / (s + EPS) for m, s in model_scores.items()}
    s_inv = sum(inv.values())
    w_data = {m: inv[m] / s_inv for m in inv}

    # 5) Clamp to [W_FLOOR, W_CAP] & re-normalize
    w_clamped = {m: float(np.clip(w, W_FLOOR, W_CAP)) for m, w in w_data.items()}
    s_c = sum(w_clamped.values())
    w_clamped = {m: w_clamped[m] / s_c for m in w_clamped}

    # 6) Optional harmful-vs-naive screen (skipped unless you supply naive HS4 sMAPE here)
    # -- place-holder: keep as-is. If you later provide a dict model -> delta_vs_naive, use it to zero-out or floor.

    # 7) Blend with prior (70% data-driven, 30% prior), then re-normalize
    w_mix = {}
    for m in w_clamped:
        w_prior = PRIOR_WEIGHTS.get(m, 0.0)
        w_mix[m] = MIX_ALPHA * w_clamped[m] + (1.0 - MIX_ALPHA) * w_prior
    s_m = sum(w_mix.values())
    w_mix = {m: w_mix[m] / s_m for m in w_mix}

    # 8) Recency tilt (+10% to winners on closest folds), then re-normalize
    winners = month_winners(hs4_all, recency_months)
    w_final = w_mix.copy()
    boosted = False
    for m in winners:
        if m in w_final:
            w_final[m] *= RECENCY_TILT
            boosted = True
    if boosted:
        s_f = sum(w_final.values())
        w_final = {m: w_final[m] / s_f for m in w_final}

    diag = {
        "segment": segment,
        "horizon": horizon,
        "model_scores_smape": model_scores,
        "w_data": w_data,
        "w_clamped": w_clamped,
        "w_mixed_prior": w_mix,
        "recency_winners": list(sorted(winners)),
        "w_final": w_final,
    }
    return w_final, diag

In [5]:
# === cell: save weights rows ===
def append_weights_csv(segment, weights, notes="inv_sMAPE+clamp+prior70_30+recency10"):
    horizon, _, _ = SEGMENTS[segment]
    rows = []
    for model, w in weights.items():
        rows.append({
            "segment": segment,
            "horizon": horizon,
            "model": model,
            "weight": float(w),
            "notes": notes,
        })
    df = pd.DataFrame(rows)
    out_path = FINAL_DIR / "blend_weights_final.csv"
    if out_path.exists():
        existing = pd.read_csv(out_path)
        # remove any previous rows for this segment to avoid duplicates
        existing = existing[existing["segment"] != segment]
        df = pd.concat([existing, df], ignore_index=True)
    df.to_csv(out_path, index=False)
    print(f"Wrote weights for {segment} -> {out_path}")

In [6]:
# === cell: blend forecasts for a segment ===
def blend_forecasts_for_segment(segment, weights):
    horizon, _, _ = SEGMENTS[segment]
    # Load all model forecast files for this segment
    forecasts = []
    for _, model in MODELS:
        df = load_forecast_file(model, segment, horizon)
        # keep consistent columns
        forecasts.append(df[["origin","destination","hs6","trade_flow","month","y_pred","model"]])
    F = pd.concat(forecasts, ignore_index=True)

    # Merge 3 columns per model onto a single table for weighted sum
    keys = ["origin","destination","hs6","trade_flow","month"]
    pivot = (F
             .pivot_table(index=keys, columns="model", values="y_pred", aggfunc="sum")
             .reset_index())
    # Fill any missing model columns with 0 (rare)
    for _, model in MODELS:
        if model not in pivot.columns:
            pivot[model] = 0.0

    # Weighted sum
    y_ens = np.zeros(len(pivot), dtype=float)
    for _, model in MODELS:
        w = weights.get(model, 0.0)
        y_ens += w * pivot[model].values
    pivot["y_pred_ensemble"] = np.clip(y_ens, 0.0, None)

    # Save HS6 ensemble
    hs6_cols = keys + ["y_pred_ensemble"]
    out_hs6 = pivot[hs6_cols].copy()
    hs6_path = FINAL_DIR / f"ensemble_{segment}_h{horizon}_hs6_final.parquet"
    out_hs6.to_parquet(hs6_path, index=False)
    print(f"Saved HS6 ensemble: {hs6_path}")

    # Aggregate to HS4
    hs6_with_model = out_hs6.rename(columns={"y_pred_ensemble":"y_pred"})
    hs4 = aggregate_hs6_to_hs4(hs6_with_model, pred_col="y_pred")
    hs4 = hs4.rename(columns={"y_pred":"y_pred_ensemble"})
    hs4_path = FINAL_DIR / f"ensemble_{segment}_h{horizon}_hs4_final.parquet"
    hs4.to_parquet(hs4_path, index=False)
    print(f"Saved HS4 ensemble: {hs4_path}")

    return hs6_path, hs4_path

In [9]:
# === cell: run all segments in the recommended order ===
PROCESS_ORDER = ["CHN_export", "CHN_import", "USA_export", "USA_import"]

all_hs4_paths = []

for seg in PROCESS_ORDER:
    print("="*80)
    print(f"Segment: {seg}")
    weights, diag = compute_segment_weights(seg)
    print("Weighted sMAPE per model:", diag["model_scores_smape"])
    print("Recency winners:", diag["recency_winners"])
    print("Final weights:", weights)
    append_weights_csv(seg, weights)
    hs6_path, hs4_path = blend_forecasts_for_segment(seg, weights)
    all_hs4_paths.append(hs4_path)

print("="*80)
print("Done computing and blending all segments.")

Segment: CHN_export
Weighted sMAPE per model: {'lgbm_rmse': 0.47787564050667214, 'xgb_log1p': 0.48107109683865773, 'xgb_tweedie': 0.45922571088030995}
Recency winners: ['xgb_tweedie']
Final weights: {'lgbm_rmse': 0.3091258372815779, 'xgb_log1p': 0.29319100602510445, 'xgb_tweedie': 0.39768315669331766}
Wrote weights for CHN_export -> /content/drive/MyDrive/ai4trade/predictions/final/blend_weights_final.csv
Saved HS6 ensemble: /content/drive/MyDrive/ai4trade/predictions/final/ensemble_CHN_export_h2_hs6_final.parquet
Saved HS4 ensemble: /content/drive/MyDrive/ai4trade/predictions/final/ensemble_CHN_export_h2_hs4_final.parquet
Segment: CHN_import
Weighted sMAPE per model: {'lgbm_rmse': 0.6252462366858746, 'xgb_log1p': 0.6323535290582595, 'xgb_tweedie': 0.6516840751699182}
Recency winners: ['lgbm_rmse']
Final weights: {'lgbm_rmse': 0.34875751115038695, 'xgb_log1p': 0.29994385606259194, 'xgb_tweedie': 0.3512986327870211}
Wrote weights for CHN_import -> /content/drive/MyDrive/ai4trade/predict

In [11]:
# === cell: concat HS4 segment files into a single parquet ===
def concat_final_hs4(all_paths, out_name="final_forecast_hs4_final.parquet"):
    dfs = [pd.read_parquet(p) for p in all_paths]
    out = pd.concat(dfs, ignore_index=True)
    out_path = FINAL_DIR / out_name
    out.to_parquet(out_path, index=False)
    print(f"Saved merged HS4 file: {out_path}")
    return out_path

# call it now to produce the merged HS4 used by 50_make_submission.ipynb
merged_hs4_path = concat_final_hs4(all_hs4_paths)

Saved merged HS4 file: /content/drive/MyDrive/ai4trade/predictions/final/final_forecast_hs4_final.parquet


In [14]:
# === cell: read and display merged HS4 file ===
merged_df = pd.read_parquet(merged_hs4_path)
display(merged_df.tail())

,origin,destination,trade_flow,month,hs4,y_pred_ensemble
85186,USA,ZAF,Import,2025-07-01,9702,1.491543e+05
85187,USA,ZAF,Import,2025-07-01,9703,5.276047e+05
85188,USA,ZAF,Import,2025-07-01,9705,8.746720e+04
85189,USA,ZAF,Import,2025-07-01,9706,7.818510e+03
85190,USA,ZAF,Import,2025-07-01,9999,1.145277e+07


In [15]:
# Calculate the total y_pred_ensemble for each segment
segment_totals = merged_df.groupby(['origin', 'trade_flow'])['y_pred_ensemble'].sum().reset_index()

# Display the results
print("Total y_pred_ensemble by Segment:")
display(segment_totals)

Total y_pred_ensemble by Segment:


,origin,trade_flow,y_pred_ensemble
0,CHN,Export,2.357066e+11
1,CHN,Import,1.385913e+11
2,USA,Export,1.278939e+11
3,USA,Import,2.198430e+11


In [16]:
# Find the number of unique partners (destination) for each country and trade_flow
partner_counts = merged_df.groupby(['origin', 'trade_flow'])['destination'].nunique().reset_index(name='num_partners')

# Display the results
print("Number of unique partners by Origin and Trade Flow:")
display(partner_counts)

Number of unique partners by Origin and Trade Flow:


,origin,trade_flow,num_partners
0,CHN,Export,30
1,CHN,Import,29
2,USA,Export,30
3,USA,Import,30
